In [ ]:
# digital_print / pdf_vector: clean digitally exported prints (PDF; TIFF later),
\n# NOT ink-on-paper scans (those stay on the scan workflow).
\n# Flow: auto regions (header / scale-legend / tracks / curves) → digitize → LAS.
\n# Run: PYTHONPATH=src python notebooks/pdf_vector_demo.py
\n# USGS horizontal example: scripts/pdf_vector_e2e_usgs.py → digital_print_usgs/.
\nfrom __future__ import annotations
\n
\nimport urllib.request
\nfrom pathlib import Path
\n
\nimport numpy as np
\n
\nfrom pylogdigitizer.pdf_vector import digitize_pdf
\nfrom pylogdigitizer.las_export import read_las, verify_las
\nfrom pylogdigitizer.qc_compare import metrics, to_grid
\n
\nDATA = Path("example_data/pdf_vector")
\nDATA.mkdir(parents=True, exist_ok=True)
\nCBL_URL = "https://gdr.openei.org/files/1330/78B-32_CBL_LOG.zip"
\nZIP = DATA / "78B-32_CBL_LOG.zip"
\nPDF_NAME = "UniversityofUtah_Forge78B-32_6Oct21_CBL.Pdf"
\n
\nif not (DATA / PDF_NAME).exists():
\n    if not ZIP.exists():
\n        print("downloading", CBL_URL)
\n        urllib.request.urlretrieve(CBL_URL, ZIP)
\n    import zipfile
\n    with zipfile.ZipFile(ZIP) as zf:
\n        for n in zf.namelist():
\n            if n.lower().endswith((".pdf", ".las")):
\n                zf.extract(n, DATA)
\n                (DATA / n).rename(DATA / Path(n).name)
\n    print("extracted to", DATA)
\n
\npdf = DATA / PDF_NAME
\n# Auto legend styles (colour+width+dash) — no hand mnemonic_colors required.
\n# Pages span shallow→deep so MTEM linear wrap unwraps by continuity.
\nlog = digitize_pdf(pdf, pages=[5, 10, 19, 29], step=0.5)
\nprint("well", log.well)
\nprint("layout axis", [L.axis for L in log.layouts])
\nprint("regions", [(r.kind, r.label) for L in log.layouts for r in L.regions[:8]])
\nprint("legend", {
\n    h.mnemonic: {"color": h.color, "width": h.width, "dash": h.dash,
\n                 "scale": f"{h.v_left}-{h.v_right} {h.unit}"}
\n    for h in log.headers if h.color is not None
\n})
\nprint("curves", {
\n    t.mnemonic: (round(float(t.depth.min()), 1), round(float(t.depth.max()), 1), int(t.info["n_pts"]))
\n    for t in log.traces
\n})
\nout = DATA / "forge78B_cbl_pages_digitized.las"
\nlog.to_las(out)
\nprint("wrote", out, "verify", verify_las(out, log.depth, log.curves) or "ok")
\nprint("lasio reload", [c.mnemonic for c in read_las(out).curves])
\n
\nlas_path = next(DATA.glob("*CBL*Main*.las"), None) or next(DATA.glob("*.las"), None)
\nif las_path and las_path.resolve() != out.resolve():
\n    import lasio
\n    df = lasio.read(str(las_path)).df()
\n    alias = {"ECGR_STGC": "GR_STGC", "TT": "TT", "CBL": "CBL", "MTEM": "MTEM"}
\n    for t in log.traces:
\n        m = alias.get(t.mnemonic)
\n        if not m or m not in df.columns or t.mnemonic not in log.curves:
\n            continue
\n        ref = df[m].replace(-999.25, np.nan).dropna()
\n        ok = np.isfinite(log.depth) & np.isfinite(log.curves[t.mnemonic])
\n        td, tv = log.depth[ok], log.curves[t.mnemonic][ok]
\n        if len(td) < 20:
\n            continue
\n        rv = to_grid(ref.index.values, ref.values, td)
\n        print(t.mnemonic, metrics(rv, tv))
\n